In [2]:
import sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import StackingClassifier

In [ ]:
file_path = "paste your file path here"

In [ ]:
feature_length = 4096  # all features have this length

feature_description = {
    'tmmx': tf.io.FixedLenFeature([feature_length], tf.float32),
    'FireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'PrevFireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'th': tf.io.FixedLenFeature([feature_length], tf.float32),
    'erc': tf.io.FixedLenFeature([feature_length], tf.float32),
    'vs': tf.io.FixedLenFeature([feature_length], tf.float32),
    'elevation': tf.io.FixedLenFeature([feature_length], tf.float32),
    'tmmn': tf.io.FixedLenFeature([feature_length], tf.float32),
    'NDVI': tf.io.FixedLenFeature([feature_length], tf.float32),
    'sph': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pdsi': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pr': tf.io.FixedLenFeature([feature_length], tf.float32),
    'population': tf.io.FixedLenFeature([feature_length], tf.float32),
}

def _parse_function(proto):
    return tf.io.parse_single_example(proto, feature_description)

dataset = tf.data.TFRecordDataset(file_path)
parsed_dataset = dataset.map(_parse_function)

# Define the list of feature names and the label name
features_to_extract = ['tmmx', 'PrevFireMask', 'th', 'erc', 'vs', 'elevation',
                       'tmmn', 'NDVI', 'sph', 'pdsi', 'pr', 'population']
label_to_extract = 'FireMask'

In [17]:
all_features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
all_labels_data = []

for parsed_record in parsed_dataset:
  for i in range(len(features_to_extract)):
    all_features_data[i].append(parsed_record[features_to_extract[i]].numpy())

    # Extract the label (FireMask) for the current recor
  all_labels_data.append(parsed_record[label_to_extract].numpy())

# Convert the lists of data into numpy arrays
all_features_data = np.array(all_features_data)

X_transposed = all_features_data.transpose(1, 2, 0)
X_final = X_transposed.reshape(-1, 12)

# Convert the lists of data into numpy arrays
X = np.array(X_final)
y = np.array(all_labels_data).flatten()

#Filtering out -1 pixels
mask = y != -1
X = X[mask]
y = y[mask]

# Sanity check
assert set(np.unique(y)).issubset({0, 1}), "Found unexpected labels!"


print(f"Shape of features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")

Shape of features (X): (4007752, 12)
Shape of labels (y): (4007752,)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(n_jobs =-1, max_depth=4)

In [ ]:
estimators = [
    ('lr', lr_model),
    ('rf', rf_model)
]

# Initialize the StackingClassifier
# The final_estimator is the model that combines the predictions of the base estimators.
stacking_classifier = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5
)

stacking_classifier.fit(X_train, y_train)

In [ ]:
test_y = y_test
test_x = X_test
predictions = stacking_classifier.predict(test_x)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

overall_precision = precision_score(test_y, predictions, average = 'macro')
overall_recall = recall_score(test_y, predictions, average = 'macro')
overall_f1 = f1_score(test_y, predictions, average = 'macro')


# Calculate accuracy
print("Accuracy:", accuracy_score(test_y, predictions))

print(f"Overall Precision: {overall_precision}")
print(f"Overall Recall: {overall_recall}")
print(f"Overall F1 Score: {overall_f1}")

print("Classification Report:\n", classification_report(test_y, predictions))

In [ ]:
#Single record prediction visualization
#For one to put on the slides, skipping 2 records is good

dataset = tf.data.TFRecordDataset(file_path)

num_records_to_skip = 2

single_record = dataset.map(_parse_function).skip(num_records_to_skip).take(1)

features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
labels_data = []

for parsed_record in single_record:
  #if parsed_record['FireMask'].numpy().mean() >= 0:
    for i in range(len(features_to_extract)):
        features_data[i].append(parsed_record[features_to_extract[i]].numpy())
    labels_data.append(parsed_record[label_to_extract].numpy())

features_data = np.array(features_data)

print(f"Shape of features_data: {features_data.shape}") 

# Shape: (n_records, 64, 64, 12)
X_transposed_single = features_data.transpose(1, 2, 0).reshape(-1, 64, 64, 12)
# Shape: (n_records, 64, 64)
y_grid_single = np.array(labels_data).reshape(-1, 64, 64)

n_records = X_transposed_single.shape[0]
X_neighbors_single = []
y_neighbors_single = []

for rec in range(n_records):
    # Exclude 1-pixel border: iterate rows 1..62, cols 1..62
    for r in range(1, 63):
        for c in range(1, 63):
            label = y_grid_single[rec, r, c]

            # Extract 3x3 neighborhood for all 12 features, then flatten
            # Shape: (3, 3, 12) -> (108,)
            neighborhood_single = X_transposed_single[rec, r-1:r+2, c-1:c+2, :]
            X_neighbors_single.append(neighborhood_single.flatten())
            y_neighbors_single.append(label)

X_single = np.array(X_neighbors_single)  # shape: (n_valid_pixels, 108)
y_single = np.array(y_neighbors_single)  # shape: (n_valid_pixels,)

#assert set(np.unique(y_single)).issubset({0, 1}), "Found unexpected labels!"
print(f"Shape of features (X): {X_single.shape}")  # e.g. (n, 108)
print(f"Shape of labels (y): {y_single.shape}")
#print(f"Label distribution: {np.bincount(y_single.astype(int))}")

Shape of features_data: (12, 1, 4096)
Shape of features (X): (3844, 108)
Shape of labels (y): (3844,)


In [ ]:
single_record_prediction = stacking_classifier.predict(X_single)

plt.figure(figsize=(10, 5))

plt.subplot(1,2,1)
plt.title(f"Prediction of Single Record FireMask")
plt.imshow(single_record_prediction.reshape(62,62), cmap="Reds")
plt.colorbar()

plt.subplot(1,2,2)
plt.title(f"Single Record FireMask")
plt.imshow(y_single.reshape(62,62), cmap="Reds")
plt.colorbar()

plt.show()